# Python 进阶特性：面向初学者的友好指南
## 基于一个餐厅排队模拟脚本

---

本笔记本将带你逐一了解一个**餐厅排队模拟程序**中运用的 Python 进阶特性。只要你掌握了 Python 基础——变量、循环、函数与简单的类——就可以跟上本教程的步伐。

### 涵盖主题

| # | 特性 | 一句话简介 |
|---|------|-----------|
| 1 | **类、类型注解与 `@dataclass`** | 理解类是什么，并学会自动生成类的样板代码 |
| 2 | **`field()` 与可变默认值** | 避开一个经典的 Python 陷阱 |
| 3 | **`datetime` 模块** | 将字符串解析为日期与时间戳 |
| 4 | **Lambda 函数** | 编写简洁的一次性匿名函数 |
| 5 | **列表推导式** | 用一行代码构建过滤列表 |
| 6 | **`for...else` 结构** | Python 独有的循环语法 |

---

---
## 1. 类、类型注解与 `@dataclass`

### 1.1 什么是类（Class）？

在日常生活中，很多事物都共享同一种"结构"。餐厅里的每位顾客都有：姓名、人数、到达时间、就餐时长……**类（Class）** 就是描述这种结构的**模板**。你可以把一个**类**想象成一张空白表格的模版，而每一个**实例（Instance）** 就是按照模版填写好的一张具体表格。

```python
# 定义一个模版（类）
class Customer:
    name:   str   # 字段 1：姓名
    people: int   # 字段 2：人数

# 按照模版创建一个实例
c = Customer()
c.name   = "Alice"
c.people = 4
```

类还可以包含**方法（Method）**——即定义在类内部、专属于该类的函数：

```python
class Customer:
    name:   str
    people: int

    def greet(self):           # self 指向当前实例本身
        print(f"你好，{self.name}！")
```

---

### 1.2 类型注解（Type Hints）

Python 是**动态类型语言**——变量的类型由 Python 自动推断，无需事先声明：

```python
age  = 25        # Python 推断：整数（int）
name = "Alice"   # Python 推断：字符串（str）
```

这很方便，但会让代码难以阅读。看到下面这个函数，你能一眼知道参数是什么类型吗？

```python
def seat(req, table, cnt, time):
    ...
```

**类型注解**让你可以*自愿地*标注变量的预期类型：

```python
# 语法：变量名: 类型 = 值
age:     int   = 25
name:    str   = "Alice"
price:   float = 9.99
is_open: bool  = True
```

对于函数，你还可以注解**返回值类型**：

```python
#        参数注解              返回类型
#           ↓                     ↓
def greet(name: str) -> str:
    return "你好，" + name
```

> **关键点：** 类型注解在运行时**不会被强制检查**——它们是给*人类和编辑器*看的提示，不是给解释器看的命令。VS Code 等 IDE 会利用这些提示在运行前帮你发现潜在的类型错误。

---

### 1.3 `@dataclass` 装饰器

如果没有 `@dataclass`，你需要为类手动编写大量重复代码：

```python
class Request:
    def __init__(self, index, people, arrival, duration):
        self.index    = index      # ← 每个字段名出现了三次！
        self.people   = people
        self.arrival  = arrival
        self.duration = duration

    def __repr__(self):
        return (f"Request(index={self.index}, people={self.people}, "
                f"arrival={self.arrival}, duration={self.duration})")
```

字段越多，重复越多，越枯燥。

**`@dataclass`** 是一个**装饰器**（`@` 语法），它会根据类中的类型注解，**自动为你生成** `__init__`、`__repr__` 和 `__eq__` 方法：

```python
from dataclasses import dataclass

@dataclass
class Request:
    index:    int
    people:   int
    arrival:  int
    duration: int
# 仅此而已！Python 自动生成了初始化、打印和比较方法
```

**设置默认值**——在类型注解后直接赋值即可：

```python
@dataclass
class Request:
    index:    int
    people:   int
    arrival:  int
    duration: int
    table:    int = -1   # -1 表示"尚未就座"
    leave:    int = 0
    share:    int = 1
```

> **规则：** 有默认值的字段必须排在无默认值字段的**后面**，与普通函数的参数规则一致。

**装饰器是什么？** 装饰器是一个包装函数或类以增加额外行为的函数。`@` 语法只是一种简写：

```python
@dataclass
class Foo:
    ...

# 等价于：
Foo = dataclass(Foo)   # dataclass() 对类进行转换
```

随着学习深入，你还会遇到更多内置装饰器：`@staticmethod`、`@classmethod`、`@property` 等。

In [ ]:
# ── 类型注解（Type Hints）演示 ───────────────────────────────────────────────

# 1. 注解普通变量
index:    int  = 1
people:   int  = 4
duration: int  = 60   # 单位：分钟
share:    bool = True

print(f"共 {people} 人，就餐 {duration} 分钟，愿意拼桌：{share}")

# 2. 注解函数参数与返回值类型
def calculate_leave_time(arrival: int, duration: int) -> int:
    """返回顾客离开的时间（到达时间 + 就餐时长）。"""
    return arrival + duration

leave = calculate_leave_time(arrival=100, duration=60)
print(f"t=100 到达，t={leave} 离开")

# 3. 类型注解不会被强制执行——仅供参考
def add(a: int, b: int) -> int:
    return a + b

# Python 允许传入字符串——注解在运行时会被忽略！
result = add("你好，", "世界")
print(result)  # 可以运行！Python 执行了字符串拼接而非数字相加

In [ ]:
# ── @dataclass 演示 ──────────────────────────────────────────────────────────

# ── 传统写法（不使用 @dataclass） ───────────────────────────────
class RequestManual:
    def __init__(self, index, people, arrival, duration, table=-1):
        self.index    = index
        self.people   = people
        self.arrival  = arrival
        self.duration = duration
        self.table    = table

    def __repr__(self):
        return (f"Request(index={self.index}, people={self.people}, "
                f"arrival={self.arrival}, duration={self.duration}, "
                f"table={self.table})")

r1 = RequestManual(index=1, people=4, arrival=0, duration=60)
print("传统写法：", r1)

In [ ]:
# ── 使用 @dataclass 的简洁写法 ──────────────────────────────────
from dataclasses import dataclass

@dataclass
class Request:
    index:    int
    people:   int
    arrival:  int
    duration: int
    table:    int = -1   # 默认：尚未就座
    leave:    int = 0    # 默认：尚未计算
    share:    int = 1    # 默认：愿意拼桌

r2 = Request(index=1, people=4, arrival=0, duration=60)
print("使用 @dataclass：", r2)

# @dataclass 还会自动生成 __eq__：
r3 = Request(index=1, people=4, arrival=0, duration=60)
print("r2 == r3？", r2 == r3)   # True——字段值相同

# 访问字段与普通类完全一样：
print(f"人数：{r2.people}，就餐时长：{r2.duration} 分钟")

---
## 2. `field()` 与可变默认值陷阱

### 问题：可变默认参数

这是 Python 中最经典的"坑"之一。假设你在类中用列表作为默认值：

```python
@dataclass
class Table:
    index: int
    customers: list = []   # ← 这是一个 Bug！
```

为什么是 Bug？因为**所有实例会共享同一个列表对象**。在 Python 中，类（和函数）定义中的默认值只会在代码加载时被求值**一次**，而不是每次创建新实例时都重新创建。

```python
t1 = Table(index=1)
t2 = Table(index=2)

t1.customers.append("Alice")
print(t2.customers)   # ['Alice'] ← 意外！t2 也被影响了！
```

两张桌子共享了内存中的**同一个**列表——修改一个就会影响另一个。

事实上，`@dataclass` 会**直接报错**，如果你试图将可变对象（list、dict、set）作为默认值——这正是因为这个 Bug 太常见了。

### 解决方案：`field(default_factory=...)`

`dataclasses` 模块中的 `field()` 函数允许你指定一个**工厂函数（factory）**——每次创建新实例时，它都会调用该函数来生成一个全新的对象：

```python
from dataclasses import dataclass, field

@dataclass
class Table:
    index:     int
    customers: list = field(default_factory=list)
    #                                       ^^^^
    # 每次创建新 Table 实例时，都会调用 list() 生成一个新列表
```

现在 `Table(1)` 和 `Table(2)` 各自拥有**独立的空列表**。

### 通用规律

| 默认值类型 | 写法 |
|-----------|------|
| 不可变值（int、str、bool）| `字段: int = 0`（直接赋默认值即可）|
| 空列表 | `字段: list = field(default_factory=list)` |
| 空字典 | `字段: dict = field(default_factory=dict)` |
| 任意自定义对象 | `字段: Foo = field(default_factory=Foo)` |

In [ ]:
# ── 可变默认值 Bug 演示 ──────────────────────────────────────────────────────
# （使用普通类——@dataclass 在导入时就会阻止这种写法）

class BuggyTable:
    def __init__(self, index, customers=[]):  # ← 所有实例共享同一个列表！
        self.index = index
        self.customers = customers

t1 = BuggyTable(index=1)
t2 = BuggyTable(index=2)

t1.customers.append("Alice")

print(f"t1.customers：{t1.customers}")
print(f"t2.customers：{t2.customers}")
print()
print("是同一个列表对象吗？", t1.customers is t2.customers)  # True——这就是 Bug！

In [ ]:
# ── 修复方法：field(default_factory=list) ────────────────────────────────────
from dataclasses import dataclass, field

@dataclass
class Table:
    index:      int
    max_people: int
    cur_people: int  = 0
    customers:  list = field(default_factory=list)  # ← 每个实例拥有自己的列表

t1 = Table(index=1, max_people=4)
t2 = Table(index=2, max_people=2)

t1.customers.append("Alice")

print(f"t1.customers：{t1.customers}")
print(f"t2.customers：{t2.customers}")
print()
print("是同一个列表对象吗？", t1.customers is t2.customers)  # False——已修复！

---
## 3. `datetime` 模块

### 为什么需要它？

真实数据通常将日期时间存储为文本字符串，例如 `"20240101120000"`（表示 2024 年 1 月 1 日 12:00:00）。要做任何时间运算——"两个事件之间相差多少分钟？"——首先需要将这些字符串解析为真正的 datetime 对象。

### `datetime.strptime()` — 将字符串解析为日期时间

**`strptime`** 是 *"string parse time"（字符串解析时间）* 的缩写。你需要提供：
1. 要解析的字符串
2. 一个**格式字符串**，说明字符串中每个部分的含义

常用格式代码：

| 代码 | 含义 | 示例 |
|------|------|------|
| `%Y` | 4 位年份 | `2024` |
| `%m` | 2 位月份 | `01` – `12` |
| `%d` | 2 位日期 | `01` – `31` |
| `%H` | 24 小时制小时 | `00` – `23` |
| `%M` | 分钟 | `00` – `59` |
| `%S` | 秒 | `00` – `59` |

```python
from datetime import datetime

dt = datetime.strptime("20240101120000", "%Y%m%d%H%M%S")
# dt 现在是一个 datetime 对象：2024-01-01 12:00:00
```

### `.timestamp()` — 转换为 Unix 时间戳

**Unix 时间戳**是从 1970 年 1 月 1 日起经过的**秒数**。它是用纯数字表示某一时刻的通用方法。

```python
seconds = dt.timestamp()   # 例如：1704096000.0
```

### 在本脚本中的应用

```python
arrival_dt = datetime.strptime(arrival, '%Y%m%d%H%M%S')
arrival_timestamp = int(arrival_dt.timestamp() / 60)   # 秒 → 分钟
```

除以 `60` 将单位从秒转换为分钟，让模拟中使用的数字更简洁。

In [ ]:
# ── datetime 演示 ────────────────────────────────────────────────────────────
from datetime import datetime

# 1. 解析时间戳字符串（CSV 文件中使用的格式）
raw_string = "20240101120000"   # 2024 年 1 月 1 日 12:00:00
dt = datetime.strptime(raw_string, "%Y%m%d%H%M%S")
print(f"解析结果：   {dt}")
print(f"年：  {dt.year}")
print(f"月：  {dt.month}")
print(f"时：  {dt.hour}")

# 2. 转换为 Unix 时间戳（自 1970-01-01 起的秒数）
unix_seconds = dt.timestamp()
print(f"\nUnix 时间戳（秒）：{unix_seconds}")

# 3. 转换为分钟（与脚本中的处理方式相同）
unix_minutes = int(unix_seconds / 60)
print(f"Unix 时间戳（分钟）：{unix_minutes}")

# 4. 模拟两位顾客并计算相对到达时间
t1 = datetime.strptime("20240101120000", "%Y%m%d%H%M%S")
t2 = datetime.strptime("20240101120930", "%Y%m%d%H%M%S")

start_min   = int(t1.timestamp() / 60)
arrive1_min = int(t1.timestamp() / 60) - start_min   # 0
arrive2_min = int(t2.timestamp() / 60) - start_min   # 9（9 分 30 秒向下取整）

print(f"\n顾客 1 相对到达时间：t = {arrive1_min} 分钟")
print(f"顾客 2 相对到达时间：t = {arrive2_min} 分钟")

---
## 4. Lambda 函数

### 什么是 Lambda？

**Lambda** 是一种**匿名**（没有名字）的小函数，可以用单行表达式来定义。它适合在只需要一次使用的场合，通常作为参数传递给其他函数。

### 语法

```
lambda <参数> : <表达式>
```

对比普通函数与 lambda：

```python
# 普通命名函数
def square(x):
    return x * x

# 等价的 lambda（匿名）
square = lambda x: x * x
```

两者调用方式完全相同：`square(5)` → `25`。

### 最常见的用途：排序中的 `key=` 参数

Python 的 `list.sort()` 和内置 `sorted()` 函数接受一个 `key` 参数——一个从每个元素中提取比较依据的函数。

```python
requests.sort(key=lambda x: x.arrival)
#                     ↑
#           "对于每个请求 x，按照其 arrival 字段排序"
```

如果不用 lambda，你需要专门写一个函数：

```python
def get_arrival(x):
    return x.arrival

requests.sort(key=get_arrival)  # 效果相同，但代码更啰嗦
```

### Lambda 的局限性

- 只能包含**单一表达式**（不能有 `if` 块、循环或多条语句）
- 最适合**简单的、一次性**操作
- 遇到复杂逻辑时，写一个正式的命名函数要清晰得多

In [ ]:
# ── Lambda 演示 ──────────────────────────────────────────────────────────────
from dataclasses import dataclass

@dataclass
class SimpleRequest:
    index:   int
    people:  int
    arrival: int   # 相对于模拟开始的分钟数

requests = [
    SimpleRequest(index=1, people=2, arrival=15),
    SimpleRequest(index=2, people=4, arrival=3),
    SimpleRequest(index=3, people=1, arrival=9),
]

print("排序前：")
for r in requests:
    print(f"  请求 {r.index}：t={r.arrival} 到达")

# 使用 lambda 按到达时间排序
requests.sort(key=lambda x: x.arrival)

print("\n按到达时间排序后：")
for r in requests:
    print(f"  请求 {r.index}：t={r.arrival} 到达")

# 多参数 lambda
add = lambda a, b: a + b
print(f"\nadd(3, 5) = {add(3, 5)}")

# 在 sorted() 中使用 lambda——按人数排序（最多的排前面）
by_size = sorted(requests, key=lambda x: x.people, reverse=True)
print("\n按人数排序（最多的优先）：")
for r in by_size:
    print(f"  请求 {r.index}：共 {r.people} 人")

---
## 5. 列表推导式

### 繁琐写法：用 `for` 循环构建列表

假设你想找出某张桌子上所有已经到该离开的顾客：

```python
leaving = []
for c in table.customers:
    if c.leave <= cur_time:
        leaving.append(c)
```

这能用，但为了表达一个简单的想法，花了四行代码。

### 列表推导式：一行优雅搞定

**列表推导式**将同样的逻辑压缩为一个表达式：

```python
leaving = [c for c in table.customers if c.leave <= cur_time]
```

从左到右阅读：*"给我 `c`，对于 `table.customers` 中的每个 `c`，但只要 `c.leave <= cur_time`"*

### 语法结构

```
[<表达式>  for <变量> in <可迭代对象>  if <条件>]
    ↑           ↑            ↑             ↑
  保留什么   你给它的名字   数据来源    可选过滤条件
```

`if` 部分是可选的：

```python
# 无过滤——对每个元素做变换
squares = [x * x for x in range(5)]           # [0, 1, 4, 9, 16]

# 有过滤——只保留部分元素
even = [x for x in range(10) if x % 2 == 0]  # [0, 2, 4, 6, 8]

# 变换 + 过滤
even_squares = [x*x for x in range(10) if x % 2 == 0]  # [0, 4, 16, 36, 64]
```

### 在本脚本中的应用

```python
leaving = [c for c in t.customers if c.leave <= cur_time]
```

这会收集所有预定离开时间已过的顾客，以便将他们从桌位上移除。

In [ ]:
# ── 列表推导式演示 ───────────────────────────────────────────────────────────
from dataclasses import dataclass

@dataclass
class Customer:
    name:  str
    leave: int   # 预计离开时间

customers = [
    Customer("Alice", leave=30),
    Customer("Bob",   leave=55),
    Customer("Carol", leave=70),
    Customer("Dave",  leave=45),
    Customer("Eve",   leave=90),
]

cur_time = 60
print(f"当前时间：t={cur_time}")

# ── 传统 for 循环写法 ──────────────────────────────────────────
leaving_verbose = []
for c in customers:
    if c.leave <= cur_time:
        leaving_verbose.append(c)

print("\n传统写法——已离开的顾客：")
for c in leaving_verbose:
    print(f"  {c.name}（t={c.leave} 离开）")

# ── 列表推导式写法（结果完全相同） ────────────────────────────
leaving = [c for c in customers if c.leave <= cur_time]

print("\n列表推导式——已离开的顾客：")
for c in leaving:
    print(f"  {c.name}（t={c.leave} 离开）")

# ── 更多示例 ───────────────────────────────────────────────────
still_here = [c.name for c in customers if c.leave > cur_time]
print(f"\n仍在桌位的顾客：{still_here}")

# 变换：收集仍在桌位顾客的剩余等待时间
remaining_wait = [(c.name, c.leave - cur_time) for c in customers if c.leave > cur_time]
print(f"各自剩余时间：{remaining_wait}")

---
## 6. `for...else` 结构

### 什么？循环也有 `else`？

Python 允许你为 `for`（或 `while`）循环附加一个 `else` 块。这很罕见，常常让初学者感到困惑。

**规则很简单：**
> `else` 块**仅在循环正常结束时运行**（即循环从未执行 `break`）。

```python
for item in collection:
    if <找到了想要的东西>:
        # 做一些处理
        break
else:
    # 只有在从未执行 break 时才运行
    # 即：遍历了所有元素，始终没有找到
    ...
```

### 一个具体例子

在所有桌子中寻找一张刚好合适的空桌：

```python
for t in tables:
    if t.cur_people == 0 and t.max_people == req.people:
        t.seat(req)   # 找到了完美的桌子！
        break
else:
    # 没有找到合适的空桌——退而求其次，尝试拼桌
    find_shared_table(req)
```

如果不用 `for...else`，你需要一个标志变量：

```python
found = False
for t in tables:
    if t.cur_people == 0 and t.max_people == req.people:
        t.seat(req)
        found = True
        break

if not found:
    find_shared_table(req)   # 逻辑相同，但更啰嗦
```

`for...else` 消除了多余的标志变量。

### `else` 什么时候不运行？

| 情况 | `else` 是否运行？ |
|------|-----------------|
| 循环自然结束（没有 `break`） | ✅ 运行 |
| 循环体执行了 `break` | ❌ 不运行 |
| 可迭代对象为空（循环体从未执行） | ✅ 运行 |

`else` 本质上是在说：*"如果我们遍历了所有选项，始终没有找到……"*

In [ ]:
# ── for...else 演示 ──────────────────────────────────────────────────────────

# 简单示例：在列表中搜索某个值
def search(items, target):
    for item in items:
        if item == target:
            print(f"找到了 {target}！")
            break
    else:
        # 只有在循环未被 break 时才运行
        print(f"未找到 {target}。")

search([1, 2, 3, 4, 5], 3)   # 找到了 3！
search([1, 2, 3, 4, 5], 9)   # 未找到 9。

In [ ]:
# ── 餐厅座位分配：for...else 实战 ───────────────────────────────────────────
from dataclasses import dataclass, field

@dataclass
class Table:
    index:      int
    max_people: int
    cur_people: int  = 0
    customers:  list = field(default_factory=list)

@dataclass
class Customer:
    name:   str
    people: int   # 人数
    table:  int = -1

tables = [
    Table(index=0, max_people=2, cur_people=2),  # 已满
    Table(index=1, max_people=4, cur_people=4),  # 已满
    Table(index=2, max_people=6, cur_people=0),  # 空桌（6 人桌）
]

req = Customer(name="Alice", people=4)  # 需要一张 4 人桌

print(f"正在为 {req.name}（{req.people} 人）寻找空桌...")

for t in tables:
    if t.cur_people == 0 and t.max_people == req.people:
        req.table = t.index
        print(f"  ✓ 完美匹配：桌 {t.index}（{t.max_people} 人桌，空桌）")
        break
else:
    # 没有找到完全匹配的空桌——尝试拼桌
    print("  未找到完全匹配的空桌，正在寻找有足够空位的桌子...")
    for t in tables:
        if t.max_people - t.cur_people >= req.people:
            req.table = t.index
            print(f"  ✓ 拼桌至桌 {t.index}（{t.max_people} 人桌，"
                  f"已坐 {t.cur_people} 人）")
            break
    else:
        print("  ✗ 暂无可用桌位——顾客需要等候。")

print(f"\n结果：{req.name} → 桌 {req.table}")

---
## 综合演示

现在我们已经逐一学习了每个特性，让我们看看它们如何在一个**简化版餐厅模拟**中协同工作。

注意每个特性各自扮演的角色：

| 代码位置 | 使用的特性 |
|---------|-----------|
| 类定义 | `@dataclass` + 类型注解 + `field()` |
| `create_request_from_row()` | `datetime.strptime()` 与 `.timestamp()` |
| `requests.sort(...)` | Lambda 函数（作为 `key`） |
| `[c for c in ... if ...]` | 带过滤条件的列表推导式 |
| `for t in tables: ... else:` | `for...else` 循环 |

In [ ]:
# ── 综合餐厅模拟（简化版） ────────────────────────────────────────────────────
from dataclasses import dataclass, field
from datetime import datetime

# ── 1. @dataclass + 类型注解 + field() ──────────────────────────────────────

@dataclass
class Request:
    index:     int
    people:    int
    arrival:   int
    duration:  int
    leave:     int = 0      # 由 seat() 计算
    share:     int = 1      # 1 = 愿意拼桌
    table:     int = -1     # -1 = 尚未就座
    wait_time: int = 0

@dataclass
class Table:
    index:      int
    max_people: int
    cur_people: int  = 0
    customers:  list = field(default_factory=list)  # ← 可变默认值，安全！

    def seat(self, req: Request, cur_time: int):
        req.table     = self.index
        req.leave     = req.arrival + req.duration
        req.wait_time = cur_time - req.arrival
        self.cur_people += req.people
        self.customers.append(req)

# ── 2. datetime：解析 CSV 时间戳 ─────────────────────────────────────────────

def create_request_from_row(row: list) -> Request:
    index, people, arrival_str, duration, share = row
    arrival_dt        = datetime.strptime(arrival_str, "%Y%m%d%H%M%S")
    arrival_timestamp = int(arrival_dt.timestamp() / 60)  # 秒 → 分钟
    return Request(
        index=int(index),
        people=int(people),
        arrival=arrival_timestamp,
        duration=int(duration),
        share=int(share),
    )

# ── 示例数据（通常从 CSV 文件读取） ──────────────────────────────────────────
raw_csv = [
    ["1", "4", "20240101120000", "60", "1"],
    ["2", "2", "20240101120500", "45", "0"],
    ["3", "2", "20240101120300", "30", "1"],
]

requests = [create_request_from_row(row) for row in raw_csv]  # 列表推导式！

tables = [
    Table(index=0, max_people=4),
    Table(index=1, max_people=2),
    Table(index=2, max_people=2),
]

# ── 3. lambda：按到达时间排序 ────────────────────────────────────────────────
requests.sort(key=lambda x: x.arrival)  # lambda 排序

# 转换为相对时间
s_time = requests[0].arrival
for req in requests:
    req.arrival -= s_time

print("按到达时间排序后的顾客请求：")
for r in requests:
    print(f"  请求 {r.index}：{r.people} 人，t={r.arrival} 到达，拼桌意愿={r.share}")

# ── 4. 模拟循环 ───────────────────────────────────────────────────────────────
print("\n── 安排就座 ──")
for req in requests:
    cur_time = req.arrival

    # 移除已离开的顾客（列表推导式 + 过滤）
    for t in tables:
        leaving = [c for c in t.customers if c.leave <= cur_time]
        for c in leaving:
            t.cur_people -= c.people
            t.customers.remove(c)

    # 寻找完全匹配的空桌（for...else）
    for t in tables:
        if t.cur_people == 0 and t.max_people == req.people:
            t.seat(req, cur_time)
            print(f"  请求 {req.index} → 安排至桌 {t.index} "
                  f"（完美匹配，等待 {req.wait_time} 分钟）")
            break
    else:
        if req.share:  # 愿意拼桌——寻找任何有足够空位的桌子
            best = None
            for t in tables:
                if t.max_people - t.cur_people >= req.people:
                    if best is None or (t.max_people - t.cur_people <
                                        best.max_people - best.cur_people):
                        best = t
            if best:
                best.seat(req, cur_time)
                print(f"  请求 {req.index} → 拼桌至桌 {best.index} "
                      f"（等待 {req.wait_time} 分钟）")
            else:
                print(f"  请求 {req.index} → 需要等候（暂无空位）")
        else:
            print(f"  请求 {req.index} → 进入等候队列（不拼桌且无空桌）")

# ── 5. 统计数据 ───────────────────────────────────────────────────────────────
seated = [r for r in requests if r.table != -1]   # 列表推导式
if seated:
    avg_wait = sum(r.wait_time for r in seated) / len(seated)
    print(f"\n已就座：{len(seated)}/{len(requests)} 位顾客")
    print(f"平均等待时间：{avg_wait:.1f} 分钟")

---
## 总结

以下是本笔记本涵盖的所有特性的快速参考：

### 特性速查表

| 特性 | 语法 | 关键点 |
|------|------|--------|
| **类型注解** | `x: int = 5` | 仅作提示——运行时不强制检查 |
| **`@dataclass`** | `@dataclass`（写在类上方） | 自动生成 `__init__`、`__repr__`、`__eq__` |
| **`field()`** | `x: list = field(default_factory=list)` | 防止可变默认值被共享 |
| **`datetime.strptime`** | `datetime.strptime(s, fmt)` | 将字符串转换为 datetime 对象 |
| **`.timestamp()`** | `dt.timestamp()` | Unix 时间戳：自 1970-01-01 起的秒数 |
| **Lambda** | `lambda x: x.arrival` | 内联函数，常用于 `key=` 等参数 |
| **列表推导式** | `[x for x in lst if cond]` | 一行代码构建过滤/变换后的列表 |
| **`for...else`** | `for ...: ... break` / `else:` | 只有在从未执行 `break` 时才运行 `else` |

### 各特性在脚本中的位置

```python
@dataclass()                           # ← @dataclass
class request:
    index: int                         # ← 类型注解
    customers: list = field(           # ← field()
        default_factory=list)

arrival_dt = datetime.strptime(        # ← datetime
    arrival, '%Y%m%d%H%M%S')

requests.sort(                         # ← lambda
    key=lambda x: x.arrival)

leaving = [c for c in                 # ← 列表推导式
    t.customers if c.leave <= cur_time]

for t in tables:                       # ← for...else
    if ...: t.seat(...); break
else:
    # 后备逻辑
```

---
*恭喜你完成了本教程！这些特性在真实的 Python 项目中随处可见。*